In [0]:
# Create environment widget
dbutils.widgets.dropdown("environment", "prod", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
from pyspark.sql.functions import row_number, lit
from pyspark.sql.window import Window

config = load_config_from_widget()

# Read tables
silver_orders = spark.table(f"{config['catalog']}.{config['schemas']['silver']}.orders_clean")
dim_customers = spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimCustomers")
dim_products = spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimProducts")

# Create fact table
fact_orders = (silver_orders
    .join(dim_customers, silver_orders.customer_id == dim_customers.customer_id, "left")
    .join(dim_products, silver_orders.product_id == dim_products.product_id, "left")
    .withColumn("FactOrderKey", row_number().over(Window.orderBy(lit(1))))
    .select("FactOrderKey", "order_id", dim_customers.DimCustomerKey, dim_products.DimProductKey, "order_date", "quantity", "total_amount")
)

# External table location
external_path = f"{config['storage']['external']['gold']}/FactOrders"
print(f"📂 Writing to: {external_path}")

# Write to external location
fact_orders.write.format("delta").mode("overwrite").save(external_path)

# Create external table
fact_table = f"{config['catalog']}.{config['schemas']['gold']}.FactOrders"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {fact_table}
    USING DELTA LOCATION '{external_path}'
""")

print(f"✅ {fact_table}: {spark.table(fact_table).count()} records")
print(f"✅ Delta logs created at: {external_path}/_delta_log/")